# Supplementary validation of click and search context signals

Contribution: controlled, retrained feature-group comparisons on the existing validation split,
plus four candidate-specific recent-click context counts. Run after Steps 1–4b.
This notebook leaves `features.parquet`, existing models, and existing predictions untouched.
It writes to `processed/step8_validation/<run>/` and refuses to overwrite an earlier run.

This is an exploratory extension: earlier notebooks already used the existing test split for
feature analysis. These results do not create a fresh confirmatory test set. Do not use the
test split to select features. These are LightGBM scalar-feature ablations, not an ablation
of the neural history encoder or a claim about the final ensemble.

Hypotheses:
1. Recorded recent clicks add information beyond the remaining numeric features.
2. Item-popularity counts add information beyond personal history and text.
3. Search-linked versus queryless clicks on the candidate add information beyond pooled clicks.
The -1 query marker means non-search interaction; it does not prove a specific homepage source.


In [1]:
# In Colab, run this once if dependencies are missing:
# %pip install lightgbm pyarrow pandas numpy
import os, json, platform, importlib.metadata
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/JDSearch/Raw Data')
except ImportError:
    DATA_DIR = Path(os.environ.get('JD_DATA_DIR', '~/Downloads/JDsearch')).expanduser()
PROCESSED = DATA_DIR / 'processed'
SEEDS = [42, 43, 44]  # use [42] for a preliminary run; report the configured count
TREES = 300  # fixed, equal budget for every comparison; no test-based tuning
BOOTSTRAPS = 2000
RUN = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT = PROCESSED / 'step8_validation' / RUN
OUT.mkdir(parents=True, exist_ok=False)


Mounted at /content/drive


## Helpers and data contracts
The historical `user` key is an anonymous row/slate key. Unknown metadata and absent history remain distinguishable. Ranking ties use item ID order plus seeded random tie-breaking, never original candidate position.

In [2]:
import numpy as np
import pandas as pd

EXCLUDED = {'user', 'wid', 'label', 'split', 'pos', 'position', 'candidate_position',
            'candidate_index', 'cat3_festival_lift', 'u_festival_share',
            'ucat3_fest_ord', 'ucat3_fest_share'}

def validate_table(d):
    required = {'user', 'wid', 'label', 'split', 'u_click_7d'}
    assert required.issubset(d), f'Missing columns: {required - set(d)}'
    assert d[['user', 'wid', 'label', 'split']].notna().all().all()
    assert d.label.isin([0, 1, 2, 3]).all()
    assert d.groupby('user').split.nunique().eq(1).all(), 'Slate crosses splits'
    assert not d.duplicated(['user', 'wid']).any(), 'Duplicate slate-item key'
    assert d.split.isin(['train', 'val']).all(), 'Test data must not enter this study'

def feature_groups(cols):
    return {
        'recent_click': [c for c in cols if 'click' in c],
        'popularity': [c for c in cols if c.startswith('i_users_') or c == 'slate_pct_pop'],
        'search_context': [c for c in cols if c == 'u_queryless_share_7d'],
    }

def click_context(events, candidates):
    """Only prior CLICK events; no candidate outcomes used. Type 0 = CLICK in Step 1."""
    assert events.age_days.ge(0).all() and np.isfinite(events.age_days).all()
    keys = pd.MultiIndex.from_frame(candidates[['user', 'wid']])
    out = {}
    for days in [1, 7]:
        for name, linked in [('search', True), ('queryless', False)]:
            m = (events.type == 0) & (events.age_days < days)
            m &= (events.qry >= 0) if linked else (events.qry == -1)
            counts = events.loc[m].groupby(['user', 'wid']).size()
            out[f'uc_click_{name}_{days}d_extra'] = counts.reindex(keys, fill_value=0).to_numpy(np.float32)
    return pd.DataFrame(out, index=candidates.index)

def slate_metrics(d, score, k=10, seed=42):
    """Graded NDCG, any-hit and purchase-hit; NaN for undefined denominators."""
    assert k > 0 and len(d) > 0
    score = np.asarray(score, dtype=float)
    assert len(d) == len(score) and np.isfinite(score).all()
    # Assign random tie keys in a canonical (slate, item) order to resist input-order leakage.
    canonical = np.lexsort((d.wid.to_numpy(), d.user.to_numpy()))
    tie = np.empty(len(d)); tie[canonical] = np.random.default_rng(seed).random(len(d))
    u, lab = d.user.to_numpy(), d.label.to_numpy(dtype=float)
    order = np.lexsort((tie, -score, u))
    us, ys = u[order], lab[order]
    start = np.r_[0, np.flatnonzero(us[1:] != us[:-1]) + 1]
    size = np.diff(np.r_[start, len(us)])
    gid = np.repeat(np.arange(len(start)), size)
    rank = np.arange(len(us)) - np.repeat(start, size)
    top = rank < k
    discount = np.where(top, 1 / np.log2(rank + 2), 0)
    dcg = np.bincount(gid, weights=(2 ** ys - 1) * discount)
    ideal = lab[np.lexsort((-lab, u))]
    idcg = np.bincount(gid, weights=(2 ** ideal - 1) * discount)
    pos = np.bincount(gid, weights=ys > 0) > 0
    buy = np.bincount(gid, weights=ys == 3) > 0
    hit = np.bincount(gid, weights=(ys > 0) & top) > 0
    hit_buy = np.bincount(gid, weights=(ys == 3) & top) > 0
    return pd.DataFrame({'ndcg10': np.divide(dcg, idcg, out=np.full(len(start), np.nan), where=idcg > 0),
                         'hr10': np.where(pos, hit.astype(float), np.nan),
                         'purchase_hr10': np.where(buy, hit_buy.astype(float), np.nan)},
                        index=pd.Index(us[start], name='user'))

def paired_interval(diff, draws=2000, seed=42):
    diff = np.asarray(diff, dtype=float)
    diff = diff[np.isfinite(diff)]
    if not len(diff):
        return dict(slates=0, delta=np.nan, lower95=np.nan, upper95=np.nan)
    rng = np.random.default_rng(seed)
    means = np.array([diff[rng.integers(len(diff), size=len(diff))].mean() for _ in range(draws)])
    lo, hi = np.quantile(means, [.025, .975])
    return dict(slates=len(diff), delta=float(diff.mean()), lower95=float(lo), upper95=float(hi))


## Load training and validation rows only
The existing split is preserved. Each slate is shuffled once, identically for all variants.
No outcomes or candidate positions enter the feature builders. Seasonality is excluded as
recommended by Step 4c. Save the exact column groups so the scope of each ablation is visible.
Loading all history for personal features follows the existing snapshot protocol; no candidate
labels are used to build the new features. This does not establish cross-user calendar ordering.


In [3]:
import pyarrow.dataset as ds
import lightgbm as lgb
dset = ds.dataset(PROCESSED / 'features.parquet')
feat = dset.to_table(filter=ds.field('split').isin(['train', 'val'])).to_pandas()
validate_table(feat)
BASE = [c for c in feat if c not in EXCLUDED and pd.api.types.is_numeric_dtype(feat[c])]
assert BASE and not any(c.lower() in {'pos', 'position', 'candidate_position', 'candidate_index'} for c in BASE)
feat = feat.sort_values(['user', 'wid'], kind='stable')
feat = feat.iloc[np.lexsort((np.random.default_rng(42).random(len(feat)), feat.user.to_numpy()))].reset_index(drop=True)
events = pd.read_parquet(PROCESSED / 'events.parquet', columns=['user', 'wid', 'type', 'age_days', 'qry'])
assert events.age_days.ge(0).all() and np.isfinite(events.age_days).all()
assert events.type.isin([0, 1, 2, 3]).all(), 'Expected Step 1 type codes'
assert events.qry.ge(-1).all()
age_bins = [0, 1, 3, 7, 30, 90, 366, np.inf]
temporal = pd.crosstab(pd.cut(events.age_days, age_bins, right=False), events.type)
temporal.columns = [dict(enumerate(['CLICK', 'CART', 'ORD', 'FLW']))[c] for c in temporal.columns]
temporal.to_csv(OUT / 'temporal_coverage.csv')
display(temporal)
new = click_context(events, feat[['user', 'wid']])
assert not set(new).intersection(feat.columns), 'New feature name already exists'
feat = pd.concat([feat, new], axis=1)
groups = feature_groups(BASE)
assert all(groups.values()), f'Missing expected ablation group: {groups}'
VARIANTS = {'baseline': BASE,
            **{'without_' + g: [c for c in BASE if c not in cols] for g, cols in groups.items()},
            'plus_click_context': BASE + list(new)}
(OUT / 'feature_manifest.json').write_text(json.dumps({'base': BASE, 'groups': groups, 'variants': VARIANTS}, indent=2))
print('Base features:', len(BASE)); print('Ablated columns:', groups)
print('New features:', list(new))
coverage = new.gt(0).mean().rename('candidate_nonzero_share').to_frame()
display(coverage); coverage.to_csv(OUT / 'new_feature_coverage.csv')
del events, new


,CLICK,CART,ORD,FLW
age_days,,,,
"[0.0, 1.0)",1918289,287228,36441,616
"[1.0, 3.0)",2534453,375715,57033,629
"[3.0, 7.0)",3846045,672214,109348,1406
"[7.0, 30.0)",107,3300950,553008,7419
"[30.0, 90.0)",0,6889193,1280095,14509
"[90.0, 366.0)",0,1203,4781358,0
"[366.0, inf)",0,1,0,0


Base features: 55
Ablated columns: {'recent_click': ['uc_click_7d', 'ubrand_click_7d', 'ucat3_click_7d', 'ucat2_click_7d', 'i_users_click_7d', 'c3_click_7d', 'c3_cart_per_click_7d', 'u_click_7d', 'u_cart_per_click_7d', 'u_days_since_click'], 'popularity': ['i_users_click_7d', 'i_users_cart_90d', 'i_users_ord_365d', 'slate_pct_pop'], 'search_context': ['u_queryless_share_7d']}
New features: ['uc_click_search_1d_extra', 'uc_click_queryless_1d_extra', 'uc_click_search_7d_extra', 'uc_click_queryless_7d_extra']


,candidate_nonzero_share
uc_click_search_1d_extra,0.002505
uc_click_queryless_1d_extra,0.002343
uc_click_search_7d_extra,0.003512
uc_click_queryless_7d_extra,0.003343


## Train matched models and evaluate within each slate
Each model sees identical training/validation slates and an equal 300-tree budget. Changing the
feature list is the only intervention within each seed. This avoids the earlier unequal-budget
comparison (a 600-tree baseline versus 300-tree ablations). All-zero slates are excluded from
ranking evaluation, as in earlier steps; record their counts. Purchase-HR is conditioned on
slates containing a purchase and is not a conversion-rate estimate.


In [4]:
frames = {}
excluded = {}
for split in ['train', 'val']:
    d = feat[feat.split == split].copy()
    keep = d.groupby('user').label.transform('max') > 0
    excluded[split] = {'slates_total': int(d.user.nunique()),
                       'slates_no_positive': int(d.loc[~keep, 'user'].nunique())}
    frames[split] = d.loc[keep].reset_index(drop=True)
tr, va = frames['train'], frames['val']
assert len(tr) and len(va)
assert set(tr.user).isdisjoint(va.user)
gtr = tr.groupby('user', sort=False).size().to_numpy()
print(excluded)
rows, val_scores = [], {}
for seed in SEEDS:
    for name, cols in VARIANTS.items():
        model = lgb.LGBMRanker(objective='lambdarank', label_gain=[0, 1, 3, 7],
            n_estimators=TREES, learning_rate=0.05, num_leaves=63, min_child_samples=200,
            subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
            random_state=seed, n_jobs=-1, verbose=-1)
        model.fit(tr[cols], tr.label, group=gtr)
        for split, d in [('train', tr), ('val', va)]:
            score = model.predict(d[cols])
            metrics = slate_metrics(d, score)
            rows.append(dict(seed=seed, variant=name, split=split, features=len(cols), **metrics.mean().to_dict()))
            if split == 'val':
                val_scores.setdefault(name, []).append(score)
        print(seed, name, 'validation NDCG:', rows[-1]['ndcg10'])
results = pd.DataFrame(rows)
results.to_csv(OUT / 'metrics_by_seed.csv', index=False)
display(results.groupby(['variant', 'split'])[['ndcg10', 'hr10', 'purchase_hr10']].agg(['mean', 'std']))


{'train': {'slates_total': 121539, 'slates_no_positive': 1458}, 'val': {'slates_total': 26144, 'slates_no_positive': 319}}
42 baseline validation NDCG: 0.4132028166724478
42 without_recent_click validation NDCG: 0.40420148802427425
42 without_popularity validation NDCG: 0.23074934866046237
42 without_search_context validation NDCG: 0.41250905351984735
42 plus_click_context validation NDCG: 0.4128043984444198
43 baseline validation NDCG: 0.4128687395786861
43 without_recent_click validation NDCG: 0.4042399276343423
43 without_popularity validation NDCG: 0.23005277935557236
43 without_search_context validation NDCG: 0.41285237658604856
43 plus_click_context validation NDCG: 0.41318707879056443
44 baseline validation NDCG: 0.413317894292622
44 without_recent_click validation NDCG: 0.4036850167411767
44 without_popularity validation NDCG: 0.23075557858052978
44 without_search_context validation NDCG: 0.4131059959696075
44 plus_click_context validation NDCG: 0.413273911372955


ndcg10                hr10            \
                                  mean       std      mean       std   
variant                split                                           
baseline               train  0.430650  0.000119  0.731048  0.000181   
                       val    0.413130  0.000233  0.715198  0.000777   
plus_click_context     train  0.431112  0.000081  0.731662  0.000221   
                       val    0.413088  0.000250  0.715366  0.000427   
without_popularity     train  0.304270  0.001880  0.555209  0.003714   
                       val    0.230519  0.000404  0.437715  0.000925   
without_recent_click   train  0.420449  0.000109  0.718590  0.000505   
                       val    0.404042  0.000310  0.702033  0.000562   
without_search_context train  0.430776  0.000075  0.731537  0.000254   
                       val    0.412822  0.000300  0.714992  0.000136   

                             purchase_hr10            
                                      mean       std  
variant                split                          
baseline               train      0.665350  0.000431  
                       val        0.650356  0.001685  
plus_click_context     train      0.664832  0.000680  
                       val        0.647400  0.001620  
without_popularity     train      0.443034  0.005843  
                       val        0.339792  0.002465  
without_recent_click   train      0.659447  0.002196  
                       val        0.642255  0.002376  
without_search_context train      0.666455  0.000564  
                       val        0.648823  0.002943

## Paired uncertainty and click-coverage segments
Compare seed-averaged predictions using paired resampling of whole validation slates.
Intervals are exploratory and unadjusted for multiple comparisons. Seed spread is reported
separately above. Segment presence describes coverage, not a causal effect of clicking.


In [5]:
averaged = {name: slate_metrics(va, np.mean(scores, axis=0)) for name, scores in val_scores.items()}
comparisons = []
for name, m in averaged.items():
    if name == 'baseline': continue
    assert m.index.equals(averaged['baseline'].index)
    for metric in ['ndcg10', 'hr10', 'purchase_hr10']:
        diff = (m[metric] - averaged['baseline'][metric]).dropna().to_numpy()
        comparisons.append(dict(variant=name, metric=metric, **paired_interval(diff, BOOTSTRAPS)))
comparison = pd.DataFrame(comparisons)
display(comparison); comparison.to_csv(OUT / 'paired_validation_comparisons.csv', index=False)
has_click = va.groupby('user').u_click_7d.first().gt(0)
segments = []
for name, m in averaged.items():
    for present in [False, True]:
        part = m.loc[has_click.reindex(m.index).eq(present)]
        segments.append(dict(variant=name, has_click_7d=present, slates=len(part),
            purchase_slates=int(part.purchase_hr10.notna().sum()), **part.mean().to_dict()))
pd.DataFrame(segments).to_csv(OUT / 'click_segments.csv', index=False)
display(pd.DataFrame(segments))
manifest = {'run': RUN, 'seeds': SEEDS, 'trees': TREES, 'bootstrap_draws': BOOTSTRAPS,
    'excluded_slates': excluded, 'python': platform.python_version(),
    'packages': {p: importlib.metadata.version(p) for p in ['pandas', 'numpy', 'pyarrow', 'lightgbm']},
    'input_files': {p.name: {'bytes': p.stat().st_size, 'mtime_ns': p.stat().st_mtime_ns}
                    for p in [PROCESSED / 'features.parquet', PROCESSED / 'events.parquet']}}
(OUT / 'run_manifest.json').write_text(json.dumps(manifest, indent=2))
print('Saved supplementary results to', OUT)


,variant,metric,slates,delta,lower95,upper95
0,without_recent_click,ndcg10,25825,-0.009109,-0.010477,-0.007687
1,without_recent_click,hr10,25825,-0.014017,-0.016960,-0.010958
2,without_recent_click,purchase_hr10,3045,-0.009195,-0.017406,-0.000985
3,without_popularity,ndcg10,25825,-0.178182,-0.182270,-0.174254
4,without_popularity,hr10,25825,-0.269274,-0.276013,-0.262304
5,without_popularity,purchase_hr10,3045,-0.306404,-0.325452,-0.288342
6,without_search_context,ndcg10,25825,-0.000086,-0.000705,0.000530
7,without_search_context,hr10,25825,-0.000194,-0.001781,0.001278
8,without_search_context,purchase_hr10,3045,-0.000985,-0.004926,0.003284
9,plus_click_context,ndcg10,25825,-0.000248,-0.000931,0.000414


,variant,has_click_7d,slates,purchase_slates,ndcg10,hr10,purchase_hr10
0,baseline,False,4235,620,0.385362,0.690201,0.620968
1,baseline,True,21590,2425,0.419053,0.720982,0.659381
2,without_recent_click,False,4235,620,0.376384,0.675797,0.601613
3,without_recent_click,True,21590,2425,0.409918,0.707040,0.652784
4,without_popularity,False,4235,620,0.189717,0.384416,0.316129
5,without_popularity,True,21590,2425,0.244297,0.458870,0.352577
6,without_search_context,False,4235,620,0.385893,0.689492,0.617742
7,without_search_context,True,21590,2425,0.418845,0.720889,0.658969
8,plus_click_context,False,4235,620,0.385117,0.689256,0.614516
9,plus_click_context,True,21590,2425,0.418804,0.720426,0.655670


Saved supplementary results to /content/drive/MyDrive/JDSearch/Raw Data/processed/step8_validation/20261031T072222957647Z


## Interpret and review before merging
- Negative delta for `without_recent_click` means the selected numeric click columns help.
  Other features can still contain click information (semantic profiles and query priors).
  Do not describe this as removing every click signal.
- `without_popularity` removes `i_users_*` and `slate_pct_pop`; category priors remain.
- `without_search_context` tests the existing queryless-share feature.
- Positive delta for `plus_click_context` suggests the four new candidate-specific counts help;
  a confidence interval crossing zero is inconclusive. Low coverage may explain a weak result.
- Record negative findings. Promote new features only after team review of validation results,
  runtime and input provenance. This notebook does not alter the final ensemble.
- Preserve output cells and the CSVs when you run it. Record the AI assistance, what you checked,
  and any revisions in the project's required AI interaction log. Report actual results only.
